# Etapa 5e — Ronda de mejoras del algoritmo (una sola)

**Pregunta.** ¿Se puede mejorar la combinación adoptada en `05b` dándole a la regresión más información, sobre todo para los vasos finos?

**Candidatas** (todas a la mitad de resolución, con el banco de mapas de `src/pipeline.py`, `compute_feature_bank`):

| Candidata | Mapas |
|---|---|
| A. Actual (`05b`) | verde corregido, sombrero de copa, detector de líneas, filtro adaptado, Gabor y Frangi, con la mayor respuesta entre escalas (6) |
| B. Escalas separadas | A más el detector de líneas, el filtro adaptado, Gabor y Frangi en cada una de sus 5 escalas por separado (26) |
| C. Con textura | A más contraste, homogeneidad y energía de la matriz de coocurrencia del verde corregido (9; prueba 4 del plan) |
| D. Todo | B más la textura (29) |

Quedan fuera, por tiempo: la resolución completa y los términos cuadráticos (se informan como mejoras posibles).

**Cómo se mide.** Igual que `05b`: regresión logística ajustada con las personas de entrenamiento de cada vuelta, fracción de píxeles elegida en entrenamiento, y medición en las personas de afuera a resolución original con todos los píxeles: Matthews, Dice de línea central y sensibilidad de línea central en vasos finos.

**Cuándo se adopta** (fijado antes de correr). Una candidata reemplaza a A solo si mejora Matthews con un intervalo al 95 % que no incluye el cero y no empeora los vasos finos. Si varias cumplen, gana la que tiene menos mapas entre las no inferiores a la mejor (margen −0,03). Si ninguna cumple, queda A. Después de esta ronda el algoritmo se cierra.


In [1]:
import sys
import time
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd

from src import config as C
from src.data import dev_ids, load_split
from src.evaluation import map_images, weighted_matthews
from src.split import person_folds
from src.stats import fit_logistic, percentile_interval, person_bootstrap_means, predict_logistic

ids = dev_ids()
dev = load_split().set_index("image").loc[ids]
folds = dev["person"].map(person_folds(dev.reset_index(), 5, C.SEED))
N_TRAIN, N_THR = 20_000, 30_000
BG_W = 1 / C.EVAL_BG_FRACTION
FRACS = np.round(np.arange(0.05, 0.20, 0.01), 2)

## Pasada 1: banco de mapas y muestras

In [2]:
def pass1(image_id):
    import numpy as np
    from src.data import load_image
    from src.evaluation import load_sample, sample_values
    from src.pipeline import compute_feature_bank
    s = load_sample(image_id)
    bank, names = compute_feature_bank(load_image(image_id))
    V = np.stack([sample_values(f, s) for f in bank], axis=1).astype(np.float32)
    rng = np.random.default_rng(C.SEED + sum(map(ord, image_id)))
    pos, neg = np.flatnonzero(s.truth), np.flatnonzero(~s.truth)
    train = np.r_[rng.choice(pos, N_TRAIN, replace=False), rng.choice(neg, N_TRAIN, replace=False)]
    thr = rng.choice(len(s.truth), N_THR, replace=False)
    return {"image": image_id, "names": names, "Xtr": V[train], "ytr": s.truth[train],
            "Xthr": V[thr], "ythr": s.truth[thr], "wthr": np.where(s.truth[thr], 1.0, BG_W)}

t = time.time()
P1 = {r["image"]: r for r in map_images(pass1, ids, workers=4)}
NAMES = P1[ids[0]]["names"]
print(f"Pasada 1: {time.time() - t:.0f} s; {len(NAMES)} mapas")
base = C.SEGMENT_FEATURES
scales = [n for n in NAMES if n[-1].isdigit()]
texture = [n for n in NAMES if n.startswith("textura")]
CANDS = {"A. actual": base, "B. escalas separadas": base + scales,
         "C. con textura": base + texture, "D. todo": base + scales + texture}
COLS = {k: [NAMES.index(n) for n in v] for k, v in CANDS.items()}
print({k: len(v) for k, v in COLS.items()})

Pasada 1: 471 s; 29 mapas
{'A. actual': 6, 'B. escalas separadas': 26, 'C. con textura': 9, 'D. todo': 29}


In [3]:
def wquantile(v, w, q):
    o = np.argsort(v); cw = np.cumsum(w[o]) / w.sum()
    return v[o][np.searchsorted(cw, q)]

fit = {}
for k in range(5):
    tr = list(folds.index[folds != k])
    X = np.concatenate([P1[i]["Xtr"] for i in tr]); y = np.concatenate([P1[i]["ytr"] for i in tr])
    for cand, cols in COLS.items():
        w = fit_logistic(X[:, cols], y, ridge=1e-3)
        sc = {i: predict_logistic(P1[i]["Xthr"][:, cols], w) for i in tr}
        frac = max(FRACS, key=lambda f: np.mean([
            weighted_matthews(sc[i] > wquantile(sc[i], P1[i]["wthr"], 1 - f), P1[i]["ythr"], P1[i]["wthr"]) for i in tr]))
        fit[(k, cand)] = (w, float(frac))
print(pd.DataFrame({c: [fit[(k, c)][1] for k in range(5)] for c in COLS}, index=[f"vuelta {k}" for k in range(5)]))

          A. actual  B. escalas separadas  C. con textura  D. todo
vuelta 0       0.09                  0.10            0.10     0.10
vuelta 1       0.09                  0.10            0.09     0.09
vuelta 2       0.09                  0.09            0.09     0.09
vuelta 3       0.09                  0.09            0.09     0.09
vuelta 4       0.09                  0.09            0.09     0.09


## Pasada 2: personas de afuera, a resolución original

In [4]:
def pass2(image_id):
    import numpy as np
    from src.config import WIDTH_LIMITS_PX
    from src.data import load_fov, load_image, load_vessels
    from src.enhance import upscale
    from src.fov import detect_fov_robust
    from src.metrics import centerline_dice, matthews
    from src.pipeline import compute_feature_bank
    from src.segment import segmentar_fraccion
    from src.stats import predict_logistic
    from src.vessels import skeleton_widths
    k = int(folds[image_id])
    rgb = load_image(image_id)
    truth, fov = load_vessels(image_id), load_fov(image_id)
    bank, _ = compute_feature_bank(rgb)
    auto = detect_fov_robust(rgb)
    skel, wd = skeleton_widths(truth & fov)
    thin = skel & (wd <= WIDTH_LIMITS_PX[0])
    flat = bank.reshape(len(bank), -1).T
    rows = []
    for cand, cols in COLS.items():
        w, frac = fit[(k, cand)]
        prob = upscale(predict_logistic(flat[:, cols], w).reshape(bank.shape[1:]).astype(np.float32), fov.shape)
        m = segmentar_fraccion(prob, auto, frac)[0]
        rows.append({"image": image_id, "candidata": cand, "matthews": matthews(m, truth, fov),
                     "dice_linea_central": centerline_dice(m, truth, fov), "finos": m[thin].mean()})
    return rows

t = time.time()
R = pd.DataFrame([r for rows in map_images(pass2, ids, workers=4) for r in rows])
R.to_csv(ROOT / "results" / "05e_ronda_mejoras.csv", index=False)
print(f"Pasada 2: {time.time() - t:.0f} s")

Pasada 2: 477 s


## Resultados

In [5]:
ORDER = list(COLS)
out = []
for meas in ("matthews", "dice_linea_central", "finos"):
    w = R.pivot(index="image", columns="candidata", values=meas)[ORDER]
    d = w.sub(w["A. actual"], axis=0)
    lo, hi = percentile_interval(person_bootstrap_means(d, dev["person"]))
    for i, c in enumerate(ORDER):
        out.append({"medida": meas, "candidata": c, "media": w[c].mean(), "vs A": d[c].mean(),
                    "inferior": lo[i], "superior": hi[i]})
table = pd.DataFrame(out).set_index(["medida", "candidata"])
table.round(4)

media    vs A  inferior  superior
medida             candidata                                               
matthews           A. actual             0.7194  0.0000    0.0000    0.0000
                   B. escalas separadas  0.7255  0.0061    0.0042    0.0080
                   C. con textura        0.7177 -0.0017   -0.0031   -0.0003
                   D. todo               0.7254  0.0060    0.0041    0.0081
dice_linea_central A. actual             0.7104  0.0000    0.0000    0.0000
                   B. escalas separadas  0.7326  0.0222    0.0182    0.0261
                   C. con textura        0.7073 -0.0031   -0.0045   -0.0018
                   D. todo               0.7284  0.0180    0.0151    0.0215
finos              A. actual             0.2631  0.0000    0.0000    0.0000
                   B. escalas separadas  0.3369  0.0739    0.0556    0.0929
                   C. con textura        0.2693  0.0062   -0.0043    0.0183
                   D. todo               0.3110  0.0479    0.0331    0.0649

In [6]:
print("Matthews por grupo:")
R.join(dev["group"], on="image").pivot_table(index="candidata", columns="group", values="matthews").round(3)

Matthews por grupo:


group,dr,g,h
candidata,,,
A. actual,0.671,0.723,0.764
B. escalas separadas,0.678,0.728,0.771
C. con textura,0.669,0.722,0.763
D. todo,0.679,0.730,0.767


## Decisión (con la regla fijada antes de correr)

| Candidata | Matthews contra A (intervalo) | Vasos finos contra A (intervalo) | ¿Cumple? |
|---|---|---|---|
| B. Escalas separadas (26) | +0,0061 (+0,0042 a +0,0080) | +0,074 (+0,056 a +0,093) | sí |
| C. Con textura (9) | −0,0017 (−0,0031 a −0,0003) | +0,006 (−0,004 a +0,018) | no |
| D. Todo (29) | +0,0060 (+0,0041 a +0,0081) | +0,048 (+0,033 a +0,065) | sí |

Cumplen B y D. La mejor es B, que además es la de menos mapas: **se adopta B** (plan, desvío 14.6) y el algoritmo queda cerrado.

- **Con B:** Matthews 0,7255, Dice de línea central 0,733 y vasos finos 34 % (contra 26 % con A).
- **Imágenes y personas:** mejora Matthews en 26 de las 30 imágenes y en 17 de las 18 personas; la imagen que más pierde baja 0,005. El Dice de línea central y los vasos finos mejoran en las 30 imágenes y en los tres grupos (finos: retinopatía 29 % → 35 %, glaucoma 28 % → 37 %, sanos 22 % → 29 %).
- **Dónde está la ganancia:** en Matthews es chica. La que importa está en los vasos finos, que eran la limitación principal (etapa 6), y en la línea central, que no depende del ancho que marcó el experto. Separar las escalas le deja a la regresión pesar más las finas, en lugar de quedarse solo con la escala que más responde en cada píxel.
- **Textura:** sobre el verde corregido no le suma a los filtros. Se informa como resultado negativo del tema obligatorio.

**Cuidado:** se eligió entre cuatro candidatas con las mismas 30 imágenes, así que el valor de B en desarrollo es algo optimista; la medida que vale es la de prueba. Con B, la fracción elegida en entrenamiento fue 0,10 en dos de las cinco vueltas: por eso las máscaras de desarrollo (`05d`) usan la fracción de cada vuelta.